# Validación de Candidatos — Exoplanetas TESS
### Segunda etapa: revisión de estrellas marcadas como `posible_transito` o `posible_segunda_senal`

**Proyecto universitario — Pipeline de validación**

Este notebook toma el CSV de resultados generado por `DetectorAutomatizado.ipynb` y te ayuda a decidir, candidato por candidato, si una señal BLS parece un tránsito planetario real o un falso positivo:

1. Carga los candidatos con `posible_transito=True` o `posible_segunda_senal=True`.
2. Descarga **todos los sectores TESS disponibles** por estrella (más datos = más confianza).
3. Corre la **prueba de tránsitos par/impar** (clásica para detectar binarias eclipsantes).
4. Busca un **eclipse secundario** en fase 0.5 (otra señal típica de binaria).
5. Si una estrella tiene señal primaria y segunda señal, comprueba si los períodos están en una **resonancia orbital** simple.
6. Consulta el **NASA Exoplanet Archive** con más detalle (disposición, método de descubrimiento).
7. Genera una figura de 4 paneles por candidato.
8. Te deja escribir un **veredicto manual** (candidato fuerte / probable falso positivo / necesita más datos) que se guarda en `candidatos_validados.csv`, sin repetir candidatos ya revisados.

---
**Estructura del notebook:**
- Sección A: Imports
- Sección B: Configuración
- Sección C: Cargar candidatos desde el CSV de resultados
- Sección D: Funciones de validación
- Sección E: Revisión interactiva (candidato por candidato)

> ⚠️ Estas pruebas son heurísticas, no una confirmación definitiva. Los resultados fuertes deben contrastarse con velocidad radial, fotometría de otro telescopio o un ajuste de modelo completo (p. ej. `batman`).

## A. Imports

In [1]:
import os
import re
import math
import warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import astropy.units as u
from astroquery.utils.tap import TapPlus

import lightkurve as lk

warnings.filterwarnings("ignore")

# Mismo estilo visual "observatorio" que el detector automático
PALETTE = {
    "bg": "#0d1117",
    "grid": "#21262d",
    "text": "#e6edf3",
    "muted": "#8b949e",
    "border": "#30363d",
    "blue": "#58a6ff",
    "red": "#f85149",
    "green": "#3fb950",
    "orange": "#f0883e",
    "gold": "#ffa657",
    "white": "#f0f6fc",
}

plt.rcParams.update({
    'figure.facecolor': PALETTE["bg"],
    'axes.facecolor': PALETTE["bg"],
    'axes.edgecolor': PALETTE["border"],
    'axes.labelcolor': PALETTE["text"],
    'xtick.color': PALETTE["muted"],
    'ytick.color': PALETTE["muted"],
    'text.color': PALETTE["text"],
    'grid.color': PALETTE["grid"],
    'grid.linestyle': '--',
    'grid.alpha': 0.5,
    'font.family': 'monospace',
    'figure.dpi': 110
})

print("✓ Bibliotecas de validación cargadas.")


✓ Bibliotecas de validación cargadas.


## B. Configuración

In [3]:
# ============================================================
# CONFIGURACIÓN
# ============================================================

CARPETA_SALIDA = "/home/gael/Documents/CosasEscuela/ProyectoCiencias"
NOMBRE_CSV_RESULTADOS = "resultados_exoplanetas.csv"   # el que genera el detector automático
NOMBRE_CSV_VALIDADOS = "candidatos_validados.csv"       # veredictos manuales de esta revisión
CARPETA_FIGURAS_VALIDACION = "figuras_validacion"

# Umbral de diferencia par/impar (%) por encima del cual se marca como
# sospechoso de binaria eclipsante.
UMBRAL_DIFERENCIA_PAR_IMPAR_PCT = 25.0

# Tolerancia para considerar dos períodos en resonancia orbital.
TOLERANCIA_RESONANCIA_PCT = 3.0

# Ancho de fase (en fracción del período) usado para promediar el fondo
# del tránsito en las pruebas par/impar y de eclipse secundario.
MEDIO_ANCHO_FASE = 0.03

print("Carpeta actual:", CARPETA_SALIDA)
print("CSV de resultados:", NOMBRE_CSV_RESULTADOS)
print("CSV de validados:", NOMBRE_CSV_VALIDADOS)


Carpeta actual: /home/gael/Documents/CosasEscuela/ProyectoCiencias
CSV de resultados: resultados_exoplanetas.csv
CSV de validados: candidatos_validados.csv


## C. Cargar candidatos desde el CSV de resultados

In [4]:
# ============================================================
# CARGA DE CANDIDATOS
# ============================================================

COLUMNAS_VALIDADOS = [
    "tic_id",
    "nombre_estrella",
    "tipo_senal",
    "periodo_dias",
    "snr",
    "profundidad_pct",
    "profundidad_par_pct",
    "profundidad_impar_pct",
    "diferencia_par_impar_pct",
    "posible_binaria_por_par_impar",
    "profundidad_eclipse_secundario",
    "posible_binaria_por_eclipse_secundario",
    "resonancia_con_otra_senal",
    "planeta_confirmado_en_archive",
    "disposicion_archive",
    "fecha_hora_revision",
    "veredicto",
    "notas",
]


def leer_csv_seguro(path, columnas=None):
    path = Path(path)
    if not path.exists():
        return pd.DataFrame(columns=columnas or [])
    try:
        return pd.read_csv(path)
    except Exception:
        return pd.DataFrame(columns=columnas or [])


def guardar_csv(path, df):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False, encoding="utf-8-sig")


def append_validado(path_csv, fila):
    df = leer_csv_seguro(path_csv, COLUMNAS_VALIDADOS)
    nueva = pd.DataFrame([fila])
    for col in COLUMNAS_VALIDADOS:
        if col not in nueva.columns:
            nueva[col] = np.nan
    nueva = nueva[COLUMNAS_VALIDADOS]
    df = pd.concat([df, nueva], ignore_index=True)
    guardar_csv(path_csv, df)
    return df


def _a_booleano(serie):
    """Convierte una columna que representa verdadero/falso a booleanos reales,
    sin importar cómo haya quedado guardada en el CSV (True/False nativos,
    texto 'True'/'FALSE'/'true', 1/0, 'si'/'no', vacíos, etc.). Cualquier
    valor no reconocido se trata como False."""
    if serie is None:
        return pd.Series(dtype=bool)
    if serie.dtype == bool:
        return serie.fillna(False)
    valores_verdaderos = {"true", "1", "1.0", "si", "sí", "yes", "verdadero", "y"}
    return serie.astype(str).str.strip().str.lower().isin(valores_verdaderos)


def cargar_candidatos(carpeta, nombre_csv_resultados):
    """Lee el CSV de resultados y arma una lista de candidatos a revisar:
    una fila por señal primaria (posible_transito) y otra por segunda señal
    (posible_segunda_senal), cuando corresponda."""
    csv_resultados = Path(carpeta) / nombre_csv_resultados
    df = leer_csv_seguro(csv_resultados)

    print(f"📄 Leyendo candidatos desde: {csv_resultados.resolve()}")
    if not csv_resultados.exists():
        print(
            "   ⚠️ Ese archivo no existe en esta carpeta. Revisa CARPETA_SALIDA "
            "y NOMBRE_CSV_RESULTADOS en la Sección B, o corre primero "
            "DetectorAutomatizado.ipynb para generarlo."
        )
    else:
        print(f"   Filas totales en el CSV: {len(df)}")

    if df.empty:
        return pd.DataFrame(), csv_resultados

    if "posible_transito" not in df.columns:
        print("   ⚠️ El CSV no tiene una columna 'posible_transito'.")
        mask_primaria = pd.Series(False, index=df.index)
    else:
        mask_primaria = _a_booleano(df["posible_transito"])

    if "posible_segunda_senal" not in df.columns:
        print("   ⚠️ El CSV no tiene una columna 'posible_segunda_senal'.")
        mask_secundaria = pd.Series(False, index=df.index)
    else:
        mask_secundaria = _a_booleano(df["posible_segunda_senal"])

    print(f"   Marcadas posible_transito=True: {mask_primaria.sum()}")
    print(f"   Marcadas posible_segunda_senal=True: {mask_secundaria.sum()}")

    if mask_primaria.sum() == 0 and mask_secundaria.sum() == 0:
        if "posible_transito" in df.columns:
            print(
                "   ℹ️ Valores únicos vistos en 'posible_transito':",
                list(df["posible_transito"].unique())[:10],
            )
        if "posible_segunda_senal" in df.columns:
            print(
                "   ℹ️ Valores únicos vistos en 'posible_segunda_senal':",
                list(df["posible_segunda_senal"].unique())[:10],
            )

    candidatos = []

    primarios = df[mask_primaria]
    for _, row in primarios.iterrows():
        candidatos.append({
            "tic_id": row.get("tic_id"),
            "nombre_estrella": row.get("nombre_estrella"),
            "tipo_senal": "primaria",
            "periodo_dias": row.get("periodo_detectado_dias"),
            "snr": row.get("snr_bls"),
            "profundidad_pct": row.get("profundidad_pct"),
            "planeta_confirmado_en_archive": row.get("planeta_confirmado_en_archive"),
            "periodo_relacionado_dias": row.get("periodo_segunda_senal_dias"),
        })

    secundarios = df[mask_secundaria]
    for _, row in secundarios.iterrows():
        candidatos.append({
            "tic_id": row.get("tic_id"),
            "nombre_estrella": row.get("nombre_estrella"),
            "tipo_senal": "secundaria",
            "periodo_dias": row.get("periodo_segunda_senal_dias"),
            "snr": row.get("snr_segunda_senal"),
            "profundidad_pct": np.nan,
            "planeta_confirmado_en_archive": row.get("planeta_confirmado_en_archive"),
            "periodo_relacionado_dias": row.get("periodo_detectado_dias"),
        })

    return pd.DataFrame(candidatos), csv_resultados


def candidatos_pendientes(candidatos_df, csv_validados):
    """Quita del listado los candidatos que ya tengan un veredicto guardado
    (mismo tic_id + tipo_senal), para no revisar dos veces lo mismo."""
    validados = leer_csv_seguro(csv_validados, COLUMNAS_VALIDADOS)
    if validados.empty or candidatos_df.empty:
        return candidatos_df

    ya_hechos = set(
        zip(
            pd.to_numeric(validados["tic_id"], errors="coerce"),
            validados["tipo_senal"].astype(str),
        )
    )
    mascara = ~candidatos_df.apply(
        lambda r: (float(r["tic_id"]), str(r["tipo_senal"])) in ya_hechos, axis=1
    )
    return candidatos_df[mascara].reset_index(drop=True)


candidatos_todos, csv_resultados_path = cargar_candidatos(CARPETA_SALIDA, NOMBRE_CSV_RESULTADOS)
csv_validados_path = Path(CARPETA_SALIDA) / NOMBRE_CSV_VALIDADOS

candidatos_pend = candidatos_pendientes(candidatos_todos, csv_validados_path)

print(f"Candidatos totales (primaria + secundaria): {len(candidatos_todos)}")
print(f"Ya revisados: {len(candidatos_todos) - len(candidatos_pend)}")
print(f"Pendientes de revisar: {len(candidatos_pend)}")

if not candidatos_pend.empty:
    display(candidatos_pend[["tic_id", "nombre_estrella", "tipo_senal", "periodo_dias", "snr", "profundidad_pct"]])


📄 Leyendo candidatos desde: /home/gael/Documents/CosasEscuela/ProyectoCiencias/resultados_exoplanetas.csv
   Filas totales en el CSV: 6991
   Marcadas posible_transito=True: 1
   Marcadas posible_segunda_senal=True: 3
Candidatos totales (primaria + secundaria): 4
Ya revisados: 1
Pendientes de revisar: 3


,tic_id,nombre_estrella,tipo_senal,periodo_dias,snr,profundidad_pct
0,79303604,HD 202269A,secundaria,3.702617,16.809922,NaN
1,953416511,TIC 953416511,secundaria,0.565261,11.817502,NaN
2,161313268,TIC 161313268,secundaria,1.208201,12.949169,NaN


## D. Funciones de validación

In [5]:
# ============================================================
# FUNCIONES DE VALIDACIÓN
# ============================================================

def _limpiar_archivos_corruptos(mensaje_error):
    """Lightkurve/astropy a veces dejan un archivo FITS a medio descargar en
    la caché local (p. ej. por una descarga interrumpida). Esta función extrae
    esa ruta del mensaje, la borra, y borra también toda la carpeta contenedora
    (para eliminar metadatos asociados) antes de reintentar."""
    import shutil
    
    rutas = re.findall(r"(\S+\.fits)", str(mensaje_error))
    borrados = []
    for ruta in rutas:
        p = Path(ruta)
        if p.exists():
            try:
                # Borrar el archivo mismo
                p.unlink()
                borrados.append(str(p))
                
                # Borrar también la carpeta padre (contiene metadatos)
                parent = p.parent
                if parent.exists() and parent != Path("/"):
                    try:
                        shutil.rmtree(parent)
                        borrados.append(f"(carpeta: {str(parent)})")
                    except OSError:
                        pass
            except OSError:
                pass
    return borrados


def descargar_multisector(tic_id, mision="TESS", reintentos=3):
    """Descarga y combina TODOS los sectores TESS disponibles para un TIC.
    Más sectores = periodograma más limpio y pruebas más confiables que con
    un único sector.

    Si un archivo en la caché local está corrupto, lo detecta, borra la carpeta
    completa del producto y reintenta la descarga. Si sigue fallando después de
    varios reintentos, sugiere omitir ese candidato."""
    try:
        coleccion = None
        ultimo_error = None
        
        for intento in range(reintentos + 1):
            try:
                # Hacer una búsqueda nueva en cada intento
                sr = lk.search_lightcurve(f"TIC {int(tic_id)}", mission=mision)
                if sr is None or len(sr) == 0:
                    return None
                
                coleccion = sr.download_all()
                ultimo_error = None
                break
                
            except Exception as e_descarga:
                ultimo_error = e_descarga
                msg_error = str(e_descarga)
                
                # Detectar si es un problema de archivo corrupto
                es_corrupto = "corrupt" in msg_error.lower() or "reading Data product" in msg_error
                
                if es_corrupto:
                    borrados = _limpiar_archivos_corruptos(e_descarga)
                    if borrados and intento < reintentos:
                        print(
                            f"⚠️ Archivo(s) en caché corrupto(s) para TIC {tic_id}, "
                            f"se eliminaron y se reintenta: {', '.join(borrados[:2])}"
                        )
                        continue
                    elif intento >= reintentos:
                        print(
                            f"⚠️ Después de {reintentos + 1} intentos, sigue fallando el descargue. "
                            f"Puede ser un problema del servidor MAST con este archivo específico. "
                            f"Intenta escribir 'omitir' para saltar este candidato."
                        )
                        break
                else:
                    # No es un problema de corrupción, solo falla la descarga
                    break

        if ultimo_error is not None:
            raise ultimo_error

        if coleccion is None or len(coleccion) == 0:
            return None

        lc = coleccion.stitch()
        lc = lc.remove_nans().remove_outliers(sigma=5)

        if len(lc) > 401:
            lc = lc.flatten(window_length=401)

        return lc
    except Exception as e:
        print(f"⚠️ Error descargando TIC {tic_id}: {e}")
        return None


def test_par_impar(lc, periodo, t0=None, medio_ancho_fase=MEDIO_ANCHO_FASE):
    """Prueba clásica de binaria eclipsante: pliega la curva al DOBLE del
    período y compara la profundidad de los tránsitos que caen en fase 0
    ('pares') contra los que caen en fase 0.5 ('impares'). En un planeta
    real ambas profundidades deben ser prácticamente iguales; una binaria
    eclipsante suele mostrar una diferencia grande (eclipse primario vs.
    secundario de tamaño distinto)."""
    try:
        plegada = lc.fold(period=2 * periodo, epoch_time=t0)
        fase = plegada.phase.value
        flujo = plegada.flux.value

        m_par = np.abs(fase) < medio_ancho_fase
        m_impar = np.abs(np.abs(fase) - 0.5) < medio_ancho_fase

        if m_par.sum() < 5 or m_impar.sum() < 5:
            return None, None, None

        prof_par = 1 - np.nanmedian(flujo[m_par])
        prof_impar = 1 - np.nanmedian(flujo[m_impar])

        base = max(abs(prof_par), abs(prof_impar), 1e-9)
        diferencia_pct = abs(prof_par - prof_impar) / base * 100

        return prof_par * 100, prof_impar * 100, diferencia_pct
    except Exception:
        return None, None, None


def chequear_eclipse_secundario(lc, periodo, t0=None, medio_ancho_fase=MEDIO_ANCHO_FASE):
    """Busca una caída de brillo en fase 0.5 del período principal. Un
    eclipse secundario claro es otra señal típica de sistema binario más
    que de planeta (aunque planetas muy calientes pueden mostrar un
    eclipse secundario débil, así que esto es solo una pista más)."""
    try:
        plegada = lc.fold(period=periodo, epoch_time=t0)
        fase = plegada.phase.value
        flujo = plegada.flux.value

        m_fuera = (np.abs(fase) > 0.25) & (np.abs(fase) < 0.45)
        m_secundario = np.abs(np.abs(fase) - 0.5) < medio_ancho_fase

        if m_fuera.sum() < 10 or m_secundario.sum() < 5:
            return None

        nivel_base = np.nanmedian(flujo[m_fuera])
        nivel_secundario = np.nanmedian(flujo[m_secundario])

        return (nivel_base - nivel_secundario) * 100  # % ; >0 = caída real
    except Exception:
        return None


def chequear_resonancia(periodo1, periodo2, tolerancia_pct=TOLERANCIA_RESONANCIA_PCT):
    """Compara dos períodos contra razones orbitales simples (1:1, 2:1,
    3:2, 5:2, 5:3, ...). Una coincidencia cercana es un indicio (no una
    prueba) de que ambas señales pertenecen a un sistema multiplanetario
    real en resonancia, en vez de ser una casualidad o un artefacto."""
    if periodo1 is None or periodo2 is None:
        return None
    if not (np.isfinite(periodo1) and np.isfinite(periodo2)) or periodo2 == 0:
        return None

    razones = [(1,1), (2,1), (1,2), (3,1), (1,3), (3,2), (2,3),
               (5,2), (2,5), (5,3), (3,5), (4,1), (1,4)]

    mejor = None
    for a, b in razones:
        esperado = periodo1 * a / b
        error_pct = abs(esperado - periodo2) / periodo2 * 100
        if error_pct <= tolerancia_pct and (mejor is None or error_pct < mejor[2]):
            mejor = (a, b, error_pct)

    return mejor  # None si no hay ninguna razón simple cercana


def consultar_exoplanet_archive_detallado(tic_id):
    """Consulta el NASA Exoplanet Archive (tabla 'ps') por disposición y
    método de descubrimiento de planetas ya catalogados en este TIC."""
    try:
        tap = TapPlus(url="https://exoplanetarchive.ipac.caltech.edu/TAP")
        query = f"""
            SELECT pl_name, soltype AS disposition, pl_orbper, pl_rade, pl_bmasse, discoverymethod
            FROM ps
            WHERE tic_id = 'TIC {int(tic_id)}'
        """
        job = tap.launch_job(query)
        tabla = job.get_results()
        return tabla.to_pandas() if tabla is not None and len(tabla) > 0 else pd.DataFrame()
    except Exception as e:
        print(f"⚠️ No se pudo consultar el Exoplanet Archive: {e}")
        return pd.DataFrame()


def graficar_validacion(lc, periodo, nombre_estrella, tic_id, tipo_senal, carpeta_figuras):
    """Genera una figura de 4 paneles centrada en las pruebas de falso
    positivo: curva completa, zoom al tránsito, comparación par/impar y
    fase completa (para ver un eventual eclipse secundario)."""
    carpeta_figuras = Path(carpeta_figuras)
    carpeta_figuras.mkdir(parents=True, exist_ok=True)

    fig, axes = plt.subplots(2, 2, figsize=(14, 9))
    fig.suptitle(
        f"Validación — {nombre_estrella} (TIC {tic_id}) — señal {tipo_senal} — P={periodo:.4f} d",
        color=PALETTE["white"], fontsize=13
    )

    # Panel 1: curva completa
    ax = axes[0, 0]
    ax.scatter(lc.time.value, lc.flux.value, s=2, color=PALETTE["blue"], alpha=0.5)
    ax.set_title("Curva de luz completa (todos los sectores)", color=PALETTE["text"])
    ax.set_xlabel("Tiempo [BTJD]")
    ax.set_ylabel("Flujo normalizado")
    ax.grid(True)

    # Panel 2: zoom al tránsito (fase plegada al período)
    ax = axes[0, 1]
    plegada = lc.fold(period=periodo)
    ax.scatter(plegada.phase.value, plegada.flux.value, s=3, color=PALETTE["orange"], alpha=0.4)
    ax.set_xlim(-0.1, 0.1)
    ax.set_title("Zoom al tránsito (fase plegada)", color=PALETTE["text"])
    ax.set_xlabel("Fase")
    ax.set_ylabel("Flujo normalizado")
    ax.grid(True)

    # Panel 3: prueba par/impar (plegado al doble del período)
    ax = axes[1, 0]
    plegada2 = lc.fold(period=2 * periodo)
    fase2 = plegada2.phase.value
    flujo2 = plegada2.flux.value
    m_par = np.abs(fase2) < 0.15
    m_impar = np.abs(np.abs(fase2) - 0.5) < 0.15
    ax.scatter(fase2[m_par], flujo2[m_par], s=3, color=PALETTE["green"], alpha=0.4, label="'Par' (fase 0)")
    ax.scatter(np.abs(fase2[m_impar]) - 0.5, flujo2[m_impar], s=3, color=PALETTE["red"], alpha=0.4, label="'Impar' (fase 0.5)")
    ax.set_xlim(-0.1, 0.1)
    ax.set_title("Prueba par/impar (superpuestos)", color=PALETTE["text"])
    ax.set_xlabel("Fase (recentrada)")
    ax.set_ylabel("Flujo normalizado")
    ax.legend(fontsize=8)
    ax.grid(True)

    # Panel 4: fase completa (buscar eclipse secundario)
    ax = axes[1, 1]
    ax.scatter(plegada.phase.value, plegada.flux.value, s=2, color=PALETTE["gold"], alpha=0.35)
    ax.axvline(0.5, color=PALETTE["red"], linestyle="--", alpha=0.6, label="Fase 0.5")
    ax.axvline(-0.5, color=PALETTE["red"], linestyle="--", alpha=0.6)
    ax.set_title("Fase completa (buscar eclipse secundario)", color=PALETTE["text"])
    ax.set_xlabel("Fase")
    ax.set_ylabel("Flujo normalizado")
    ax.legend(fontsize=8)
    ax.grid(True)

    fig.tight_layout()

    nombre_archivo = f"validacion_TIC{tic_id}_{tipo_senal}.png"
    ruta = carpeta_figuras / nombre_archivo
    fig.savefig(ruta, dpi=130, facecolor=PALETTE["bg"])
    plt.show()
    plt.close(fig)

    return ruta


def interpretar_resultado(diferencia_par_impar_pct, profundidad_eclipse_secundario_pct, resonancia):
    """Traduce los números de las pruebas en un mensaje corto y legible."""
    avisos = []

    if diferencia_par_impar_pct is not None:
        if diferencia_par_impar_pct >= UMBRAL_DIFERENCIA_PAR_IMPAR_PCT:
            avisos.append(
                f"⚠️ Diferencia par/impar alta ({diferencia_par_impar_pct:.1f}%) "
                "-> sugiere binaria eclipsante, no planeta."
            )
        else:
            avisos.append(
                f"✅ Diferencia par/impar baja ({diferencia_par_impar_pct:.1f}%) "
                "-> consistente con un tránsito planetario."
            )

    if profundidad_eclipse_secundario_pct is not None:
        if profundidad_eclipse_secundario_pct > 0.02:
            avisos.append(
                f"⚠️ Posible eclipse secundario detectado "
                f"(caída ≈ {profundidad_eclipse_secundario_pct:.3f}%)."
            )
        else:
            avisos.append("✅ No se detecta un eclipse secundario claro.")

    if resonancia is not None:
        a, b, err = resonancia
        avisos.append(
            f"🔁 Posible resonancia orbital {a}:{b} con la otra señal "
            f"(error {err:.2f}%)."
        )

    return avisos


print("✓ Funciones de validación listas.")


✓ Funciones de validación listas.


## E. Revisión interactiva (candidato por candidato)

Ejecuta la siguiente celda para recorrer los candidatos **pendientes** (los que aún no tengan
veredicto en `candidatos_validados.csv`). Para cada uno:

1. Descarga todos los sectores TESS disponibles.
2. Corre la prueba par/impar y busca un eclipse secundario.
3. Si la estrella tiene señal primaria y secundaria, comprueba resonancia entre ambas.
4. Consulta el NASA Exoplanet Archive.
5. Genera y guarda la figura de 4 paneles.
6. Te pide escribir un **veredicto** y, opcionalmente, una nota. El veredicto se guarda de inmediato,
   así que puedes interrumpir la revisión en cualquier momento (Kernel → Interrupt Kernel) y retomarla
   después sin perder lo ya hecho.

Veredictos sugeridos (puedes escribir el que prefieras):
- `candidato fuerte`
- `probable falso positivo`
- `necesita mas datos`
- `redescubrimiento conocido`

In [ ]:
# ============================================================
# REVISIÓN INTERACTIVA
# ============================================================

def ejecutar_revision_candidatos():
    pendientes = candidatos_pendientes(candidatos_todos, csv_validados_path)

    if pendientes.empty:
        print("No hay candidatos pendientes de revisar. 🎉")
        return

    carpeta_figuras = Path(CARPETA_SALIDA) / CARPETA_FIGURAS_VALIDACION

    print(f"Candidatos pendientes: {len(pendientes)}")
    print("Para saltar un candidato sin guardar veredicto: escribe 'saltar'.")
    print("Para detener la revisión: Kernel → Interrupt Kernel, o escribe 'salir'.\n")

    for _, cand in pendientes.iterrows():
        tic_id = int(cand["tic_id"])
        nombre = cand["nombre_estrella"]
        tipo_senal = cand["tipo_senal"]
        periodo = float(cand["periodo_dias"])
        snr = cand.get("snr", np.nan)
        profundidad_pct = cand.get("profundidad_pct", np.nan)
        periodo_relacionado = cand.get("periodo_relacionado_dias", np.nan)

        print("\n" + "=" * 70)
        print(f"▶ TIC {tic_id} — {nombre} — señal {tipo_senal}")
        print(f"  Período: {periodo:.4f} d | SNR: {snr} | Profundidad: {profundidad_pct}%")
        print("=" * 70)

        lc = descargar_multisector(tic_id)

        fila = {
            "tic_id": tic_id,
            "nombre_estrella": nombre,
            "tipo_senal": tipo_senal,
            "periodo_dias": periodo,
            "snr": snr,
            "profundidad_pct": profundidad_pct,
            "profundidad_par_pct": np.nan,
            "profundidad_impar_pct": np.nan,
            "diferencia_par_impar_pct": np.nan,
            "posible_binaria_por_par_impar": np.nan,
            "profundidad_eclipse_secundario": np.nan,
            "posible_binaria_por_eclipse_secundario": np.nan,
            "resonancia_con_otra_senal": "",
            "planeta_confirmado_en_archive": cand.get("planeta_confirmado_en_archive", np.nan),
            "disposicion_archive": "",
            "fecha_hora_revision": datetime.now().astimezone().isoformat(),
            "veredicto": "",
            "notas": "",
        }

        if lc is None:
            print("⚠️ No se pudieron descargar datos TESS para este TIC. Se omiten las pruebas.")
            veredicto = input("\nEscribe 'omitir' para saltar este candidato sin guardar, o veredicto igual: ").strip()
            if veredicto.lower() == "omitir":
                print("Candidato omitido, no se guardó veredicto.")
                continue
            # Si escribió algo diferente, usa eso como veredicto
        else:
            print(f"Datos descargados: {len(lc)} puntos.")

            prof_par, prof_impar, diff_pct = test_par_impar(lc, periodo)
            eclipse_sec = chequear_eclipse_secundario(lc, periodo)
            resonancia = chequear_resonancia(periodo, periodo_relacionado)

            fila["profundidad_par_pct"] = prof_par
            fila["profundidad_impar_pct"] = prof_impar
            fila["diferencia_par_impar_pct"] = diff_pct
            fila["posible_binaria_por_par_impar"] = (
                bool(diff_pct is not None and diff_pct >= UMBRAL_DIFERENCIA_PAR_IMPAR_PCT)
            )
            fila["profundidad_eclipse_secundario"] = eclipse_sec
            fila["posible_binaria_por_eclipse_secundario"] = (
                bool(eclipse_sec is not None and eclipse_sec > 0.02)
            )
            if resonancia is not None:
                a, b, err = resonancia
                fila["resonancia_con_otra_senal"] = f"{a}:{b} (error {err:.2f}%)"

            archive = consultar_exoplanet_archive_detallado(tic_id)
            if not archive.empty and "disposition" in archive.columns:
                fila["disposicion_archive"] = "; ".join(sorted(set(archive["disposition"].dropna().astype(str))))

            avisos = interpretar_resultado(diff_pct, eclipse_sec, resonancia)
            print("\nResultados de las pruebas:")
            for a in avisos:
                print(" ", a)

            try:
                graficar_validacion(lc, periodo, nombre, tic_id, tipo_senal, carpeta_figuras)
            except Exception as e:
                print(f"⚠️ No se pudo generar la figura: {e}")

        veredicto = input("\nVeredicto para este candidato: ").strip()

        if veredicto.lower() == "salir":
            print("Revisión detenida por el usuario. Puedes retomarla más tarde.")
            break

        if veredicto.lower() == "saltar":
            print("Candidato omitido, no se guardó veredicto.")
            continue

        notas = input("Notas (opcional): ").strip()

        fila["veredicto"] = veredicto
        fila["notas"] = notas

        append_validado(csv_validados_path, fila)
        print(f"✓ Guardado en {csv_validados_path.name}")


# Al ejecutar esta celda comienza la revisión automáticamente.
ejecutar_revision_candidatos()


Candidatos pendientes: 3
Para saltar un candidato sin guardar veredicto: escribe 'saltar'.
Para detener la revisión: Kernel → Interrupt Kernel, o escribe 'salir'.


▶ TIC 79303604 — HD 202269A — señal secundaria
  Período: 3.7026 d | SNR: 16.809922106053694 | Profundidad: nan%
⚠️ Archivo(s) en caché corrupto(s) para TIC 79303604, se eliminaron y se reintenta: /home/gael/.lightkurve/cache/mastDownload/HLSP/hlsp_t16_tess_ffi_s0001-cam2-ccd4-06478682085518522880_tess_v01/hlsp_t16_tess_ffi_s0001-cam2-ccd4-06478682085518522880_tess_v01_llc.fits, (carpeta: /home/gael/.lightkurve/cache/mastDownload/HLSP/hlsp_t16_tess_ffi_s0001-cam2-ccd4-06478682085518522880_tess_v01)
⚠️ Archivo(s) en caché corrupto(s) para TIC 79303604, se eliminaron y se reintenta: /home/gael/.lightkurve/cache/mastDownload/HLSP/hlsp_t16_tess_ffi_s0001-cam2-ccd4-06478682085518522880_tess_v01/hlsp_t16_tess_ffi_s0001-cam2-ccd4-06478682085518522880_tess_v01_llc.fits, (carpeta: /home/gael/.lightkurve/cache/mastDownload/HLSP/hlsp_

---
## Resumen de las pruebas incluidas

| Prueba | Qué detecta | Columna en `candidatos_validados.csv` |
|---|---|---|
| Par/impar | Binarias eclipsantes (profundidades distintas cada dos tránsitos) | `diferencia_par_impar_pct`, `posible_binaria_por_par_impar` |
| Eclipse secundario | Caída de brillo en fase 0.5 (típica de binarias) | `profundidad_eclipse_secundario`, `posible_binaria_por_eclipse_secundario` |
| Resonancia | Relación simple entre dos períodos del mismo sistema | `resonancia_con_otra_senal` |
| NASA Exoplanet Archive | Si ya existe un planeta catalogado en ese TIC y su disposición | `disposicion_archive`, `planeta_confirmado_en_archive` |

### Siguientes pasos si un candidato queda como "candidato fuerte"
1. Ajustar un modelo de tránsito completo con `batman` (Mandel & Agol 2002) en vez del modelo caja de BLS.
2. Buscar observaciones de seguimiento (otro telescopio, otra misión) para confirmar el período.
3. Si es posible, solicitar o buscar datos de velocidad radial para estimar la masa.
4. Documentar el candidato con todos los campos de `resultados_exoplanetas.csv` y `candidatos_validados.csv` antes de reportarlo.

---
*Desarrollado con Python 3 · lightkurve · astropy · astroquery · pandas · matplotlib*